In [13]:
import pandas as pd
import psycopg2
from psycopg2 import sql
import os

# Database connection parameters
DB_HOST = os.getenv('DB_HOST', 'localhost')  # 'postgres' if in Docker, 'localhost' if local
DB_PORT = os.getenv('DB_PORT', '5432')
DB_USER = 'airflow'
DB_PASSWORD = 'airflow'
DB_NAME = 'airflow'

print(f"Connecting to PostgreSQL at {DB_HOST}:{DB_PORT}...")

Connecting to PostgreSQL at localhost:5432...


In [14]:
try:
    # Create connection
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME
    )
    cursor = conn.cursor()
    
    # Get list of tables
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
    """)
    tables = cursor.fetchall()
    
    print("✓ Connection successful!")
    print(f"\nAvailable tables in '{DB_NAME}':")
    for table in tables:
        print(f"  - {table[0]}")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nIf running locally, you may need to start Docker containers first:")
    print("  docker compose up -d")

✓ Connection successful!

Available tables in 'airflow':
  - log
  - dag_priority_parsing_request
  - job
  - callback_request
  - import_error
  - dag_bundle
  - dag_bundle_team
  - team
  - asset_alias
  - asset_alias_asset
  - asset
  - asset_alias_asset_event
  - asset_event
  - asset_trigger
  - trigger
  - asset_active
  - connection
  - variable
  - dag
  - slot_pool
  - dag_schedule_asset_name_reference
  - dag_schedule_asset_uri_reference
  - dag_schedule_asset_alias_reference
  - dag_schedule_asset_reference
  - task_outlet_asset_reference
  - task_inlet_asset_reference
  - asset_dag_run_queue
  - dag_version
  - dag_tag
  - dag_owner_attributes
  - dag_warning
  - dag_favorite
  - log_template
  - dag_run
  - backfill
  - dag_code
  - serialized_dag
  - dagrun_asset_event
  - task_instance
  - deadline
  - backfill_dag_run
  - dag_run_note
  - hitl_detail
  - task_map
  - task_reschedule
  - xcom
  - task_instance_note
  - task_instance_history
  - rendered_task_instance_fie

In [15]:
def load_table(table_name, limit=None):
    """Load data from a PostgreSQL table into a pandas DataFrame"""
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        
        query = f"SELECT * FROM {table_name}"
        if limit:
            query += f" LIMIT {limit}"
        
        df = pd.read_sql_query(query, conn)
        conn.close()
        
        print(f"✓ Loaded {len(df)} rows from '{table_name}'")
        return df
    
    except Exception as e:
        print(f"✗ Error loading table: {e}")
        return None

# Load a specific table (modify 'table_name' as needed)
# Example: df = load_table('your_table_name', limit=100)
print("Use: df = load_table('table_name') to load data")

Use: df = load_table('table_name') to load data


In [16]:
# Load the clean_tayara table
print("📊 Loading clean_tayara table...")

df_tayara = load_table('clean_tayara')

if df_tayara is not None:
    print(f"\n✓ Successfully loaded clean_tayara")
    print(f"   Shape: {df_tayara.shape[0]} rows × {df_tayara.shape[1]} columns")
    print(f"\n   Columns: {list(df_tayara.columns)}")
else:
    print("✗ Failed to load clean_tayara table")

📊 Loading clean_tayara table...


C:\Users\AzizMaas1999\AppData\Local\Temp\ipykernel_24904\170082937.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


✓ Loaded 9623 rows from 'clean_tayara'

✓ Successfully loaded clean_tayara
   Shape: 9623 rows × 43 columns

   Columns: ['id', 'title', 'description', 'location', 'location_finale', 'transaction_type', 'transaction_type_final', 'price_num', 'superficie_num', 'superficie_finale', 'nbr_chambres_num', 'nbr_chambres_final', 'nbr_sdb_num', 'nbr_sdb_final', 'image_count', 'pub_hours_ago', 'price_per_m2', 'scraped_at', 'published_at_from_id', 'num_etage', 'adresse_raw', 'ville', 'quartier', 'delegation', 'gouvernorat', 'matched_state', 'matched_delegation', 'matched_locality_name', 'matched_postal_code', 'latitude', 'longitude', 'geo_match_level', 'geo_source', 'type_bien_extrait', 'usage_extrait', 'titre_foncier_extrait', 'source_extraction', 'extraction_confidence', 'is_price_suspicious', 'is_surface_suspicious', 'is_city_missing', 'is_transaction_missing', 'is_geo_missing']


In [17]:
# Display first few rows of clean_tayara
print("\n" + "="*80)
print("📋 CLEAN_TAYARA - FIRST ROWS")
print("="*80)
print(df_tayara.head(10).to_string())

print("\n" + "="*80)
print("📊 DATA TYPES")
print("="*80)
print(df_tayara.dtypes)


📋 CLEAN_TAYARA - FIRST ROWS
                         id                                                                            title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              description  location location_finale transaction_type transaction_type_final  price_num  superficie_num  superficie_finale  nbr_chambres_num  nbr_chambres_final  nbr_sdb_num  nbr_sdb_final  image_count  pub_hours_ago  price_per_m2                        scraped_at       published_at_from_id num_etage            adresse_raw         ville quartier delegation gouvernorat matched_state

In [18]:
# Basic statistics for clean_tayara
print("\n" + "="*80)
print("📊 BASIC STATISTICS - CLEAN_TAYARA")
print("="*80)
print(df_tayara.describe().to_string())


📊 BASIC STATISTICS - CLEAN_TAYARA
          price_num  superficie_num  superficie_finale  nbr_chambres_num  nbr_chambres_final   nbr_sdb_num  nbr_sdb_final  image_count  pub_hours_ago  price_per_m2     latitude    longitude  extraction_confidence
count  8.546000e+03    7.455000e+03        8485.000000      6.803000e+03        6.913000e+03  7.283000e+03   7.300000e+03  9623.000000    7229.000000  7.660000e+03  4265.000000  4265.000000            9623.000000
mean   3.717052e+05    1.718224e+04         492.323349      1.864872e+02        1.835670e+02  8.336088e+03   8.316679e+03     6.354983       8.808549  5.907589e+03    36.531924    10.409498               0.547490
std    3.879034e+06    9.460672e+05        2708.303740      1.353820e+04        1.343006e+04  6.804069e+05   6.796142e+05     3.678932       6.337198  4.475117e+04     0.550171     0.364448               0.108751
min    1.000000e-01   -1.470000e+02           1.000000     -5.000000e+00       -5.000000e+00 -4.000000e+00  -4.00

In [19]:
# Data Quality Analysis for clean_tayara
print("\n" + "="*80)
print("🔍 DATA QUALITY - CLEAN_TAYARA")
print("="*80)

print(f"\n📌 Missing Values:")
missing = df_tayara.isnull().sum()
if missing.sum() == 0:
    print("   ✓ No missing values found")
else:
    print(f"   {'Column':<30} {'Count':<10} {'Percentage'}")
    print("   " + "-"*50)
    for col, count in missing[missing > 0].items():
        pct = (count / len(df_tayara)) * 100
        print(f"   {col:<30} {count:<10} {pct:>6.1f}%")

print(f"\n📌 Duplicate Rows:")
duplicates = df_tayara.duplicated().sum()
print(f"   {duplicates} duplicate rows ({(duplicates/len(df_tayara)*100):.2f}%)")

print(f"\n📌 Column Statistics:")
for col in df_tayara.columns:
    unique_count = df_tayara[col].nunique()
    print(f"   {col}: {unique_count} unique values")


🔍 DATA QUALITY - CLEAN_TAYARA

📌 Missing Values:
   Column                         Count      Percentage
   --------------------------------------------------
   transaction_type               1995         20.7%
   transaction_type_final         521           5.4%
   price_num                      1077         11.2%
   superficie_num                 2168         22.5%
   superficie_finale              1138         11.8%
   nbr_chambres_num               2820         29.3%
   nbr_chambres_final             2710         28.2%
   nbr_sdb_num                    2340         24.3%
   nbr_sdb_final                  2323         24.1%
   pub_hours_ago                  2394         24.9%
   price_per_m2                   1963         20.4%
   num_etage                      8246         85.7%
   adresse_raw                    8634         89.7%
   quartier                       8728         90.7%
   delegation                     9602         99.8%
   gouvernorat                    9101       

In [20]:
# Export clean_tayara to file
from pathlib import Path

def export_clean_tayara(file_format='csv', output_dir='data_exports'):
    """Export clean_tayara table to file"""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    try:
        if file_format == 'csv':
            file_path = output_path / 'clean_tayara.csv'
            df_tayara.to_csv(file_path, index=False)
        elif file_format == 'json':
            file_path = output_path / 'clean_tayara.json'
            df_tayara.to_json(file_path, orient='records', indent=2)
        elif file_format == 'parquet':
            file_path = output_path / 'clean_tayara.parquet'
            df_tayara.to_parquet(file_path, index=False)
        else:
            print(f"Unsupported format: {file_format}")
            return
        
        print(f"✓ Exported clean_tayara ({len(df_tayara)} rows) to {file_path}")
        return file_path
        
    except Exception as e:
        print(f"✗ Export failed: {e}")


# Example usage (uncomment to run):
# export_clean_tayara(file_format='csv')
# export_clean_tayara(file_format='json')
# export_clean_tayara(file_format='parquet')

print("📌 Use: export_clean_tayara('csv') to export to CSV file")

📌 Use: export_clean_tayara('csv') to export to CSV file


## clean_tayara Table - Quick Commands

### Access Data
```python
df_tayara  # Full DataFrame
df_tayara.head()  # First 5 rows
df_tayara.tail()  # Last 5 rows
df_tayara.shape  # (rows, columns)
df_tayara.info()  # Data types and missing values
```

### Data Analysis
```python
df_tayara.describe()  # Statistical summary
df_tayara.columns  # List all columns
df_tayara['column_name'].value_counts()  # Unique values
df_tayara.isnull().sum()  # Missing values per column
```

### Filtering & Selection
```python
df_tayara[df_tayara['column'] == value]  # Filter rows
df_tayara[['col1', 'col2']]  # Select columns
df_tayara.loc[0:10]  # Select rows by position
```

### Export
```python
export_clean_tayara('csv')  # Export as CSV
export_clean_tayara('json')  # Export as JSON
export_clean_tayara('parquet')  # Export as Parquet
```

In [21]:
# Column-by-column analysis
print("\n" + "="*80)
print("🔬 DETAILED COLUMN ANALYSIS - CLEAN_TAYARA")
print("="*80)

from pandas.api.types import is_numeric_dtype

for column in df_tayara.columns:
    dtype = df_tayara[column].dtype
    missing_count = df_tayara[column].isnull().sum()
    missing_pct = (missing_count / len(df_tayara)) * 100
    
    print(f"\n📍 {column}")
    print(f"   Type: {dtype}")
    print(f"   Missing: {missing_count} ({missing_pct:.2f}%)")
    
    # Check if column is numeric
    if is_numeric_dtype(df_tayara[column]):
        print(f"   Min: {df_tayara[column].min()}")
        print(f"   Max: {df_tayara[column].max()}")
        print(f"   Mean: {df_tayara[column].mean():.2f}")
    else:
        print(f"   Unique values: {df_tayara[column].nunique()}")
        print(f"   Most common:")
        for val, count in df_tayara[column].value_counts().head(5).items():
            print(f"      - {val}: {count}")

print("\n" + "="*80)


🔬 DETAILED COLUMN ANALYSIS - CLEAN_TAYARA

📍 id
   Type: str
   Missing: 0 (0.00%)
   Unique values: 9623
   Most common:
      - 66c48f6e860508809cffb92c: 1
      - 66e1cc974cf688ad1c4ed395: 1
      - 66eedab3b3c14db342d49ef9: 1
      - 6711244e1cde9a414fa9438c: 1
      - 67123a5e4fca0ca2a8ad5693: 1

📍 title
   Type: str
   Missing: 0 (0.00%)
   Unique values: 8717
   Most common:
      - Terrain à vendre: 27
      - أرض للبيع: 23
      - terrain a vendre: 17
      - ارض للبيع: 16
      - terrain: 15

📍 description
   Type: str
   Missing: 0 (0.00%)
   Unique values: 5900
   Most common:
      - : 3353
      - Contact : /: 117
      - 🔑 : Bureau s1
📍 : Hammam lif GP1
💸 : Prix 500 DT
⏰ : contactez nous vite pour le visiter gratuitement
📞 :: 8
      - A vendre un lot de terrain de 273 m² à Nakhil Ezzahra
Prix 335000D Négociable
☎: 8
      - لكراء في سوسة باليوم و الاسبوع: 8

📍 location
   Type: str
   Missing: 0 (0.00%)
   Unique values: 33
   Most common:
      - Tunis: 3041
      - A